In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib

# Load data
df = pd.read_csv('dataset/netflix_user_behavior_dataset.csv')

# 1. Preprocessing
# Drop non-predictive column
df_ml = df.drop(columns=['user_id'])

# Encode categorical columns
categorical_cols = ['gender', 'country', 'subscription_type', 'payment_method', 'primary_device', 'favorite_genre']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_ml[col] = le.fit_transform(df_ml[col])
    label_encoders[col] = le

# Map target variable
df_ml['churned'] = df_ml['churned'].map({'No': 0, 'Yes': 1})

# Define features and target
X = df_ml.drop('churned', axis=1)
y = df_ml['churned']

# 2. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Train Model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 4. Evaluation
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

# 5. Feature Importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values(by='importance', ascending=False)

# Save the model and encoders for the "Engineering" phase
joblib.dump(rf_model, 'churn_model.pkl')
joblib.dump(label_encoders, 'label_encoders.pkl')

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n", report)
print("\nTop 5 Important Features:\n", feature_importance.head(5))

Accuracy: 0.8007

Classification Report:
               precision    recall  f1-score   support

           0       0.80      1.00      0.89      8007
           1       0.00      0.00      0.00      1993

    accuracy                           0.80     10000
   macro avg       0.40      0.50      0.44     10000
weighted avg       0.64      0.80      0.71     10000


Top 5 Important Features:
                       feature  importance
10     avg_watch_time_minutes    0.091735
16  recommendation_click_rate    0.084881
13            completion_rate    0.080886
17      days_since_last_login    0.079037
3          account_age_months    0.077263


d:\Project\Netflix User Behaviour\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Project\Netflix User Behaviour\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Project\Netflix User Behaviour\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Train with class balancing
rf_balanced = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_balanced.fit(X_train, y_train)

# Evaluation
y_pred_bal = rf_balanced.predict(X_test)
accuracy_bal = accuracy_score(y_test, y_pred_bal)
report_bal = classification_report(y_test, y_pred_bal)

# Get feature importance for the balanced model
feature_importance_bal = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_balanced.feature_importances_
}).sort_values(by='importance', ascending=False)

# Save the improved model
joblib.dump(rf_balanced, 'churn_model_balanced.pkl')

print(f"Balanced Accuracy: {accuracy_bal:.4f}")
print("\nBalanced Classification Report:\n", report_bal)
print("\nTop 5 Important Features (Balanced):\n", feature_importance_bal.head(5))

Balanced Accuracy: 0.8007

Balanced Classification Report:
               precision    recall  f1-score   support

           0       0.80      1.00      0.89      8007
           1       0.00      0.00      0.00      1993

    accuracy                           0.80     10000
   macro avg       0.40      0.50      0.44     10000
weighted avg       0.64      0.80      0.71     10000


Top 5 Important Features (Balanced):
                       feature  importance
10     avg_watch_time_minutes    0.091284
16  recommendation_click_rate    0.084701
13            completion_rate    0.082535
17      days_since_last_login    0.079718
3          account_age_months    0.077745


d:\Project\Netflix User Behaviour\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Project\Netflix User Behaviour\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Project\Netflix User Behaviour\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

In [10]:
# Compare averages between churned and not churned
comparison = df.groupby('churned').mean(numeric_only=True)
print(comparison)

# Check unique values in target
print(df['churned'].value_counts())

               age  account_age_months  monthly_fee  devices_used  \
churned                                                             
No       40.995554           29.932286    12.333566      1.998227   
Yes      40.913689           29.639803    12.282252      2.002810   

         avg_watch_time_minutes  watch_sessions_per_week  \
churned                                                    
No                   155.156284                 9.996503   
Yes                  154.103673                 9.948414   

         binge_watch_sessions  completion_rate  rating_given  \
churned                                                        
No                   7.009641        64.551953      3.000445   
Yes                  6.973906        64.458952      3.009835   

         content_interactions  recommendation_click_rate  \
churned                                                    
No                  24.354881                  49.595514   
Yes                 24.111501                